In [1]:
import pandas as pd
import glob
import re
import os

# Consolidating Village-Level Disaster Statistics

This notebook finds all CSV files matching `* สถิติการเกิดสาธารณภัยรายหมู่บ้าน.csv` and merges them into a single cleaned dataset.

**Note on Headers:**
- Years 2565, 2566, 2567: Dual headers (Row 1: English, Row 2: Thai). We keep English.
- Other years: Single English header.

**Note on Column Order:**
- The 'ปี' (Year) column is inserted at the second position (index 1).

In [2]:
# List all relevant files
file_pattern = "* สถิติการเกิดสาธารณภัยรายหมู่บ้าน.csv"
all_files = sorted(glob.glob(file_pattern))

print(f"Found {len(all_files)} files.")

Found 11 files.


In [3]:
dataframes = []

for file_path in all_files:
    # Extract year from filename
    year_match = re.search(r'(\d+)', os.path.basename(file_path))
    year = int(year_match.group(1)) if year_match else 0
    
    print(f"Processing Year {year}...")
    
    # Handle dual headers for 2565-2567
    if year in [2565, 2566, 2567]:
        # skiprows=[1] removes the second row (Thai headers) while keeping the first row as column names
        df = pd.read_csv(file_path, skiprows=[1], low_memory=False)
    else:
        df = pd.read_csv(file_path, low_memory=False)
    
    # Clean column names
    df.columns = [c.strip() for c in df.columns]
    
    # Drop completely empty rows or footer rows (where Province is missing)
    if 'Province' in df.columns:
        df = df.dropna(subset=['Province'])
    
    # Insert Year at the second position (index 1)
    df.insert(1, 'ปี', year)
    
    dataframes.append(df)

# Merge all into one
master_df = pd.concat(dataframes, ignore_index=True)

print(f"Consolidation complete. Total records: {len(master_df)}")

Processing Year 2557...
Processing Year 2558...
Processing Year 2559...
Processing Year 2560...
Processing Year 2561...
Processing Year 2562...
Processing Year 2563...
Processing Year 2564...
Processing Year 2565...
Processing Year 2566...
Processing Year 2567...
Consolidation complete. Total records: 209935


In [ ]:
# List of numeric columns that should be filled with 0 if NaN
numeric_cols = [
    'Affected People', 'Affected Households', 'Evacuated People', 'Evacuated Households',
    'Deaths', 'Missing', 'Injured', 'Housing Damage', 'Business Damage',
    'Agriculture Damage', 'Livestock Damage', 'Fishing Damage', 'Transport Damage',
    'Health Damage', 'Culture Damage', 'Education/Sports', 'Utilities Damage',
    'Govt Property Damage', 'Other Public Benefits_1', 'Other Public Benefits_2'
]

for col in numeric_cols:
    if col in master_df.columns:
        # Convert to numeric, errors='coerce' turns non-numeric strings into NaN
        master_df[col] = pd.to_numeric(master_df[col], errors='coerce').fillna(0)

# Final fill for any remaining NaNs in other columns if necessary
master_df = master_df.fillna('')

# Save to CSV with UTF-8 BOM for Excel compatibility
output_file = 'master_village_disaster_stat_2557_2567.csv'
master_df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"Master file saved to: {output_file}")


Master file saved to: master_village_disaster_stat_2557_2567.csv


,Incident Name,ปี,Disaster Type,Disaster Date,Province Code,Province,District Code,District,Subdistrict Code,Subdistrict,...,Livestock Damage,Fishing Damage,Transport Damage,Health Damage,Culture Damage,Education/Sports,Utilities Damage,Govt Property Damage,Other Public Benefits_1,Other Public Benefits_2
0,ปี 2557 ภัยแล้ง,2557,ภัยแล้ง,17/2/2014,33,ศรีสะเกษ,3322,ศิลาลาด,332204,โจดม่วง,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,ปี 2557 ภัยแล้ง,2557,ภัยแล้ง,17/2/2014,33,ศรีสะเกษ,3322,ศิลาลาด,332204,โจดม่วง,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,ปี 2557 ภัยแล้ง,2557,ภัยแล้ง,17/2/2014,33,ศรีสะเกษ,3322,ศิลาลาด,332204,โจดม่วง,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,ปี 2557 ภัยแล้ง,2557,ภัยแล้ง,24/2/2014,60,นครสวรรค์,6005,บรรพตพิสัย,600511,หนองตางู,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,ปี 2557 ภัยแล้ง,2557,ภัยแล้ง,24/2/2014,60,นครสวรรค์,6005,บรรพตพิสัย,600511,หนองตางู,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
master_df

,Incident Name,ปี,Disaster Type,Disaster Date,Province Code,Province,District Code,District,Subdistrict Code,Subdistrict,...,Livestock Damage,Fishing Damage,Transport Damage,Health Damage,Culture Damage,Education/Sports,Utilities Damage,Govt Property Damage,Other Public Benefits_1,Other Public Benefits_2
0,ปี 2557 ภัยแล้ง,2557,ภัยแล้ง,17/2/2014,33,ศรีสะเกษ,3322,ศิลาลาด,332204,โจดม่วง,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,ปี 2557 ภัยแล้ง,2557,ภัยแล้ง,17/2/2014,33,ศรีสะเกษ,3322,ศิลาลาด,332204,โจดม่วง,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,ปี 2557 ภัยแล้ง,2557,ภัยแล้ง,17/2/2014,33,ศรีสะเกษ,3322,ศิลาลาด,332204,โจดม่วง,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,ปี 2557 ภัยแล้ง,2557,ภัยแล้ง,24/2/2014,60,นครสวรรค์,6005,บรรพตพิสัย,600511,หนองตางู,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,ปี 2557 ภัยแล้ง,2557,ภัยแล้ง,24/2/2014,60,นครสวรรค์,6005,บรรพตพิสัย,600511,หนองตางู,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
209930,ปี 2567 อัคคีภัย,2567,อัคคีภัย,31/12/2024,32,สุรินทร์,3209,ศีขรภูมิ,320914,ตรมไพร,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
209931,ปี 2567 อัคคีภัย,2567,อัคคีภัย,31/12/2024,32,สุรินทร์,3215,ศรีณรงค์,321504,หนองแวง,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
209932,ปี 2567 อัคคีภัย,2567,อัคคีภัย,31/12/2024,47,สกลนคร,4716,เจริญศิลป์,471601,บ้านเหล่า,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
209933,ปี 2567 อัคคีภัย,2567,อัคคีภัย,31/12/2024,65,พิษณุโลก,6501,เมืองพิษณุโลก,650106,ท่าโพธิ์,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
